# TTA-UC現象のGKSL-Lindblad量子ダイナミクス：包括的比較

このノートブックでは、TTA-UC（Triplet-Triplet Annihilation Upconversion）現象の
GKSL-Lindblad量子ダイナミクスシミュレーションを6つのシナリオで実行し、結果を比較します。

## シナリオ一覧

| シナリオ | 手法 | ボソン | シミュレータ |
|----------|------|--------|------------|
| 1 | 古典ODE (RK45) | なし | ClassicalGKSLSimulator |
| 2 | 古典ODE (BDF) | あり | ClassicalGKSLBosonSimulator |
| 3 | Qubit Stinespring+Trotter | なし | QubitGKSLSimulator |
| 4 | Qubit Stinespring+Trotter | あり | QubitGKSLBosonSimulator |
| 5 | Qudit Stinespring+Trotter | なし | QuditGKSLSimulator |
| 6 | Qudit Stinespring+Trotter | あり | QuditGKSLBosonSimulator |

## GKSL-Lindblad方程式

$$\frac{d\hat{\rho}}{dt} = -\frac{i}{\hbar}[\hat{H}, \hat{\rho}] + \sum_\alpha \left( \hat{L}_\alpha \hat{\rho} \hat{L}_\alpha^\dagger - \frac{1}{2}\{\hat{L}_\alpha^\dagger \hat{L}_\alpha, \hat{\rho}\}\right)$$

## 1. 共通パラメータの設定

In [ ]:
import sys
import os
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from gksl_physical_parameters import GKSLPhysicalParameters

# Non-boson parameters (4 molecules, d=3, dim=81)
params = GKSLPhysicalParameters(
    E_T=1.5,    # Triplet energy (eV)
    E_S=3.0,    # Singlet energy (eV)
    V=0.1,      # Transfer coupling (eV)
    gamma_TTA=0.05,   # TTA rate
    Gamma_fl=0.01,    # Fluorescence rate
    Gamma_ph=1e-6,    # Phosphorescence rate
    k_IC=0.005,       # Internal conversion rate
    k_ISC_ST=0.003,   # Intersystem crossing S->T
    k_ISC_TS=1e-5,    # Intersystem crossing T->S
)

# Boson parameters (2 molecules for tractable computation)
params_boson = GKSLPhysicalParameters(
    N_molecules=2, with_boson=True, n_max=1,
    omega_ph=0.15, g_eph=0.02,
)

# Simulation settings
t_max = 100.0
n_steps = 100

# For quantum simulators (Stinespring+Trotter),
# more steps are needed for accuracy but increase computation time.
# Adjust n_steps_quantum as needed.
n_steps_quantum = 100

print(f'Hilbert space dim (non-boson): {params.get_hilbert_space_dim()}')
print(f'Hilbert space dim (boson, N=2, n_max=1): {params_boson.get_hilbert_space_dim()}')
print(f'Validation: {params.validate()}')

## 2. シナリオ1: 古典GKSL（ボソン無し）

scipy.integrate.solve_ivp (RK45) による直接密度行列ODE積分。
81×81次元の密度行列を直接積分します。

In [ ]:
from classical_gksl_simulator import ClassicalGKSLSimulator
from gksl_visualization import plot_population_dynamics, plot_entropy_and_purity

sim1 = ClassicalGKSLSimulator(params)
result1 = sim1.simulate(t_max=t_max, n_steps=n_steps, initial_state='edge_triplet')

print(f'Elapsed time: {result1["elapsed_time"]:.2f}s')
print(f'Final populations: N_S0={result1["populations"][-1]["N_S0"]:.4f}, '
      f'N_T1={result1["populations"][-1]["N_T1"]:.4f}, '
      f'N_S1={result1["populations"][-1]["N_S1"]:.4f}')
print(f'Trace conservation: max|Tr-1| = {max(abs(t-1) for t in result1["trace"]):.2e}')

plot_population_dynamics(result1, title='Scenario 1: Classical GKSL (No Boson)')
plot_entropy_and_purity(result1, title='Scenario 1: Entropy & Purity')

## 3. シナリオ5: Qudit GKSL（ボソン無し）

ネイティブqutrit(d=3) + Stinespring dilation + 2次対称Trotter分解。
4 qutrit + 26 ancilla qubit = 30量子ビット相当。
禁止状態なし（qutritの自然なエンコーディング）。

In [ ]:
from qudit_gksl_simulator import QuditGKSLSimulator

sim5 = QuditGKSLSimulator(params)
result5 = sim5.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state='edge_triplet')

print(f'Elapsed time: {result5["elapsed_time"]:.2f}s')
print(f'System qutrits: {result5["n_system_qudits"]}')
print(f'Ancilla qubits: {result5["n_ancilla_qubits"]}')
print(f'Estimated gates/step: {result5["estimated_gates_per_step"]}')
print(f'Total estimated gates: {result5["total_estimated_gates"]}')
print(f'Trace conservation: max|Tr-1| = {max(abs(t-1) for t in result5["trace"]):.2e}')

plot_population_dynamics(result5, title='Scenario 5: Qudit GKSL (No Boson)')

## 4. シナリオ3: Qubit GKSL（ボソン無し）

2-qubitエンコーディング(|00⟩=S0, |01⟩=T1, |10⟩=S1, |11⟩=禁止) + Stinespring + Trotter。
8 system qubit + 26 ancilla = 34 qubit。

In [ ]:
from qubit_gksl_simulator import QubitGKSLSimulator

sim3 = QubitGKSLSimulator(params)
result3 = sim3.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state='edge_triplet')

print(f'Elapsed time: {result3["elapsed_time"]:.2f}s')
print(f'System qubits: {result3["n_sys_qubits"]}')
print(f'Ancilla qubits: {result3["n_ancilla"]}')
print(f'Total qubits: {result3["n_total_qubits"]}')
print(f'Trace conservation: max|Tr-1| = {max(abs(t-1) for t in result3["trace"]):.2e}')

plot_population_dynamics(result3, title='Scenario 3: Qubit GKSL (No Boson)')

## 5. シナリオ2: 古典GKSL（ボソン有り）

Holstein型電子-フォノン結合を含むBDF法ODE積分。
計算負荷のため、2分子・n_max=1で実行（dim=36）。

In [ ]:
from classical_gksl_boson_simulator import ClassicalGKSLBosonSimulator

sim2 = ClassicalGKSLBosonSimulator(params_boson)
result2 = sim2.simulate(t_max=t_max, n_steps=n_steps, initial_state='edge_triplet')

print(f'Elapsed time: {result2["elapsed_time"]:.2f}s')
print(f'Trace conservation: max|Tr-1| = {max(abs(t-1) for t in result2["trace"]):.2e}')

plot_population_dynamics(result2, title='Scenario 2: Classical GKSL (With Boson, N=2)')

## 6. シナリオ6: Qudit GKSL（ボソン有り）

qutrit電子系 + qutritフォノン系 + Stinespring dilation。
2分子・n_max=1で実行。

In [ ]:
from qudit_gksl_boson_simulator import QuditGKSLBosonSimulator

sim6 = QuditGKSLBosonSimulator(params_boson)
result6 = sim6.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state='edge_triplet')

print(f'Elapsed time: {result6["elapsed_time"]:.2f}s')
print(f'Trace conservation: max|Tr-1| = {max(abs(t-1) for t in result6["trace"]):.2e}')

plot_population_dynamics(result6, title='Scenario 6: Qudit GKSL (With Boson, N=2)')

## 7. シナリオ4: Qubit GKSL（ボソン有り）

Qubit Stinespring + Trotter in 拡張空間。
2分子・n_max=1で実行。

In [ ]:
from qubit_gksl_boson_simulator import QubitGKSLBosonSimulator

sim4 = QubitGKSLBosonSimulator(params_boson)
result4 = sim4.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state='edge_triplet')

print(f'Elapsed time: {result4["elapsed_time"]:.2f}s')
print(f'Trace conservation: max|Tr-1| = {max(abs(t-1) for t in result4["trace"]):.2e}')

plot_population_dynamics(result4, title='Scenario 4: Qubit GKSL (With Boson, N=2)')

## 追補: 1トロッターステップ量子回路の可視化

各量子シミュレーションで1トロッターステップに実際に適用している回路（または等価な行列演算ゲート列）を全て表示します。

In [ ]:
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate

from qubit_gksl_circuit_simulator import QubitGKSLCircuitSimulator
from qudit_gksl_circuit_simulator import QuditGKSLCircuitSimulator
from qudit_gksl_circuit_boson_simulator import QuditGKSLCircuitBosonSimulator
from stinespring_utils import stinespring_unitary_from_lindblad


def _embed_unitary_to_power_of_two(U: np.ndarray, target_dim: int) -> np.ndarray:
    if U.shape[0] != U.shape[1]:
        raise ValueError("Unitary must be square")
    if U.shape[0] > target_dim:
        raise ValueError("Target dimension is smaller than unitary dimension")

    embedded = np.eye(target_dim, dtype=np.complex128)
    embedded[: U.shape[0], : U.shape[1]] = U
    return embedded


def _print_qiskit_subcircuits(title: str, circuits: list[tuple[str, QuantumCircuit]]) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(f"subcircuits: {len(circuits)}")
    for idx, (name, circuit) in enumerate(circuits, start=1):
        print(f"\n[{idx:02d}] {name}")
        print(circuit.draw(output="text", fold=140))


def _print_qudit_subcircuits(title: str, circuits: list[tuple[str, object]]) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(f"subcircuits: {len(circuits)}")
    for idx, (name, circuit) in enumerate(circuits, start=1):
        print(f"\n[{idx:02d}] {name}")
        print(circuit.to_qasm())


dt_quantum = t_max / n_steps_quantum

# Scenario 3: Qubit GKSL (no boson)
qubit_circuit_sim = QubitGKSLCircuitSimulator(params)
qubit_step_info = qubit_circuit_sim.build_full_trotter_step_circuit(dt_quantum)
_print_qiskit_subcircuits(
    "Scenario 3: Qubit GKSL (No Boson) - 1 Trotter Step",
    qubit_step_info["circuits"],
)

# Scenario 5: Qudit GKSL (no boson)
qudit_circuit_sim = QuditGKSLCircuitSimulator(params)
qudit_step_info = qudit_circuit_sim.build_full_trotter_step_circuit(dt_quantum)
_print_qudit_subcircuits(
    "Scenario 5: Qudit GKSL (No Boson) - 1 Trotter Step",
    qudit_step_info["circuits"],
)

# Scenario 6: Qudit GKSL (with boson)
qudit_boson_circuit_sim = QuditGKSLCircuitBosonSimulator(params_boson)
qudit_boson_step_circuits: list[tuple[str, object]] = []

h1, _ = qudit_boson_circuit_sim.build_hamiltonian_circuit(dt_quantum / 2)
qudit_boson_step_circuits.append(("hamiltonian_half_1", h1))

for idx, (op_type, sites, L_local, _gamma) in enumerate(qudit_boson_circuit_sim.lindblad_local_info):
    if op_type == "single":
        circ, _ = qudit_boson_circuit_sim.build_stinespring_circuit_single(
            L_local, dt_quantum, sites[0]
        )
        qudit_boson_step_circuits.append((f"stinespring_single_{idx}", circ))
    elif op_type == "pair":
        circ, _ = qudit_boson_circuit_sim.build_stinespring_circuit_pair(
            L_local, dt_quantum, sites[0], sites[1]
        )
        qudit_boson_step_circuits.append((f"stinespring_pair_{idx}", circ))

h2, _ = qudit_boson_circuit_sim.build_hamiltonian_circuit(dt_quantum / 2)
qudit_boson_step_circuits.append(("hamiltonian_half_2", h2))

_print_qudit_subcircuits(
    "Scenario 6: Qudit GKSL (With Boson) - 1 Trotter Step",
    qudit_boson_step_circuits,
)

# Scenario 4: Qubit GKSL (with boson)
# The simulator evolves density matrices directly in the extended Hilbert space.
# Here we visualize the exact per-step operations as equivalent unitary gates.
n_sys_qubits_boson = sim4.n_el_qubits + sim4.n_ph_qubits
dim_sys_boson = 2 ** n_sys_qubits_boson

qubit_boson_step_circuits: list[tuple[str, QuantumCircuit]] = []

U_half = expm(-1j * sim4.H_total * (dt_quantum / 2))
U_half_embedded = _embed_unitary_to_power_of_two(U_half, dim_sys_boson)

ham1 = QuantumCircuit(n_sys_qubits_boson)
ham1.append(UnitaryGate(U_half_embedded, label="H_half"), range(n_sys_qubits_boson))
qubit_boson_step_circuits.append(("hamiltonian_half_1", ham1))

for idx, (L_op, _gamma) in enumerate(sim4.lindblad_ops):
    U_stinespring = stinespring_unitary_from_lindblad(L_op, dt_quantum)
    U_stinespring_embedded = _embed_unitary_to_power_of_two(
        U_stinespring, 2 * dim_sys_boson
    )

    channel_circuit = QuantumCircuit(n_sys_qubits_boson + 1)
    channel_circuit.append(
        UnitaryGate(U_stinespring_embedded, label=f"D_{idx}"),
        range(n_sys_qubits_boson + 1),
    )
    qubit_boson_step_circuits.append((f"stinespring_{idx}", channel_circuit))

ham2 = QuantumCircuit(n_sys_qubits_boson)
ham2.append(UnitaryGate(U_half_embedded, label="H_half"), range(n_sys_qubits_boson))
qubit_boson_step_circuits.append(("hamiltonian_half_2", ham2))

_print_qiskit_subcircuits(
    "Scenario 4: Qubit GKSL (With Boson) - 1 Trotter Step",
    qubit_boson_step_circuits,
)



## 8. 全シナリオの包括的比較

ボソン無しの3シナリオ（同一パラメータ）とボソン有りの3シナリオ（同一パラメータ）を
それぞれ比較します。

In [ ]:
from gksl_visualization import compare_multiple_scenarios

# Non-boson comparison (all use same 81-dim Hilbert space)
results_nb = {
    'Classical': result1,
    'Qubit': result3,
    'Qudit': result5,
}
compare_multiple_scenarios(results_nb, title='Non-Boson Scenarios Comparison (N=4)')

# Boson comparison (N=2, n_max=1)
results_b = {
    'Classical': result2,
    'Qubit': result4,
    'Qudit': result6,
}
compare_multiple_scenarios(results_b, title='Boson Scenarios Comparison (N=2, n_max=1)')

## 9. ユニタリ vs GKSL-Lindblad 比較

全散逸率を0にした純ユニタリ発展（閉量子系）と、
散逸ありのGKSL-Lindblad発展（開量子系）を比較します。

In [ ]:
from gksl_visualization import plot_gksl_comparison

# Unitary limit (all dissipation rates = 0)
params_unitary = GKSLPhysicalParameters(
    gamma_TTA=0, Gamma_fl=0, Gamma_ph=0,
    k_IC=0, k_ISC_ST=0, k_ISC_TS=0,
)
sim_unitary = ClassicalGKSLSimulator(params_unitary)
result_unitary = sim_unitary.simulate(t_max=t_max, n_steps=n_steps, initial_state='edge_triplet')

print(f'Unitary: max entropy = {max(result_unitary["entropy"]):.2e} (should be ~0)')
print(f'GKSL: final entropy = {result1["entropy"][-1]:.4f}')

plot_gksl_comparison(result_unitary, result1, title='Unitary vs GKSL-Lindblad')

## 10. 検証と考察

In [ ]:
from gksl_visualization import plot_trace_conservation

# Trace conservation check for all scenarios
print('=== Trace conservation |Tr[ρ]-1| ===')
for name, result in [('Classical NB', result1), ('Classical B', result2),
                     ('Qubit NB', result3), ('Qubit B', result4),
                     ('Qudit NB', result5), ('Qudit B', result6)]:
    max_dev = max(abs(t - 1.0) for t in result['trace'])
    print(f'  {name}: {max_dev:.2e}')

print()
print('=== Particle conservation |N_total - N_molecules| ===')
for name, result in [('Classical NB', result1), ('Qubit NB', result3),
                     ('Qudit NB', result5)]:
    N_mol = 4
    max_dev = max(abs(p['N_S0']+p['N_T1']+p['N_S1']-N_mol) for p in result['populations'])
    print(f'  {name}: {max_dev:.2e}')

print()
print('=== Computation time ===')
for name, result in [('Classical NB', result1), ('Classical B', result2),
                     ('Qubit NB', result3), ('Qubit B', result4),
                     ('Qudit NB', result5), ('Qudit B', result6)]:
    print(f'  {name}: {result["elapsed_time"]:.2f}s')

# Trace conservation plot for classical reference
plot_trace_conservation(result1, title='Trace Conservation (Classical NB)')

## 11. Stinespring忠実度の評価

古典ODE結果（理想値）とStinespring+Trotter結果の量子忠実度を計算します。

$$F(\hat{\rho}, \hat{\sigma}) = \left(\mathrm{Tr}\sqrt{\sqrt{\hat{\rho}}\hat{\sigma}\sqrt{\hat{\rho}}}\right)^2$$

In [ ]:
def quantum_fidelity(rho, sigma):
    """Compute quantum state fidelity via eigendecomposition."""
    evals, evecs = np.linalg.eigh(rho)
    evals = np.maximum(evals, 0.0)
    sqrt_rho = evecs @ np.diag(np.sqrt(evals)) @ evecs.conj().T
    M = sqrt_rho @ sigma @ sqrt_rho
    M = (M + M.conj().T) / 2
    evals_M = np.linalg.eigvalsh(M)
    evals_M = np.maximum(evals_M, 0.0)
    return float(np.real(np.sum(np.sqrt(evals_M))) ** 2)

# Compare classical vs quantum final density matrices (non-boson)
F_qudit = quantum_fidelity(result1['rho_final'], result5['rho_final'])
F_qubit = quantum_fidelity(result1['rho_final'], result3['rho_final'])

print(f'Fidelity (Classical vs Qudit): F = {F_qudit:.6f}')
print(f'Fidelity (Classical vs Qubit): F = {F_qubit:.6f}')
print(f'Fidelity (Qubit vs Qudit):     F = {quantum_fidelity(result3["rho_final"], result5["rho_final"]):.6f}')

## 12. まとめ

### 各シナリオの特徴

| シナリオ | Hilbert空間次元 | 量子資源 | 推定ゲート数/ステップ | 特徴 |
|----------|---------------|---------|---------------------|------|
| 1: Classical NB | 81 | - | - | 基準実装（厳密ODE積分） |
| 2: Classical B | 36 (N=2,n=1) | - | - | 電子-フォノン結合モデル |
| 3: Qubit NB | 81 | 34 qubits | ~194 | 2-qubitエンコード、禁止状態あり |
| 4: Qubit B | 拡張 | 42 qubits | ~194+ | ボソンモード含む |
| 5: Qudit NB | 81 | 4 qutrits + 26 ancilla | ~33 | ネイティブd=3、禁止状態なし |
| 6: Qudit B | 拡張 | 8 qutrits + 26 ancilla | ~33+ | qutritフォノンモード |

### 設計原則

- **ヒューリスティック・フォールバックの不使用**: 密度行列の強制正規化、固有値クリッピング等は一切不使用
- **厳密な物理的検証**: トレース保存、正定値性、粒子数保存を各ステップで検証
- **全エラーは例外として送出**: PhysicsViolationError, NumericalInstabilityError等